# 02 - Feature Engineering

Leakage-safe customer features and out-of-fold target encoding.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [2]:
from insurance_propensity.config import DATA_RAW_DIR, TARGET_COLUMN
from insurance_propensity.data.validation import load_raw_datasets
from insurance_propensity.features.engineering import InsuranceFeatureBuilder, CrossValidatedTargetEncoder

bundle = load_raw_datasets(DATA_RAW_DIR)
train = bundle.train
x = train.drop(columns=[TARGET_COLUMN])
y = train[TARGET_COLUMN]

builder = InsuranceFeatureBuilder().fit(x)
feature_frame = builder.transform(x.head(10_000))
feature_frame.head()

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,age_group,premium_band,vintage_band,vehicle_age_encoded,vehicle_damage_flag,not_previously_insured_flag,high_premium_flag,customer_risk_segment,cross_sell_opportunity_segment
0,1,Male,44,1,28,0,> 2 Years,Yes,40454.0,26,217,36-45,Upper Core,181-240d,2,1,1,1,Protection Gap with Damage History,Priority
1,2,Male,76,1,3,0,1-2 Year,No,33536.0,26,183,66+,Core,181-240d,1,0,1,0,Uninsured Multi-Year Vehicle,Nurture
2,3,Male,47,1,28,0,> 2 Years,Yes,38294.0,26,27,46-55,Upper Core,0-60d,2,1,1,0,Protection Gap with Damage History,Priority
3,4,Male,21,1,11,1,< 1 Year,No,28619.0,152,203,18-25,Core,181-240d,0,0,0,0,Existing Vehicle Cover Signal,Low
4,5,Female,29,1,41,1,< 1 Year,No,27496.0,152,39,26-35,Core,0-60d,0,0,0,0,Existing Vehicle Cover Signal,Low


In [3]:
encoder = CrossValidatedTargetEncoder(columns=["Region_Code", "Policy_Sales_Channel"])
encoded_sample = encoder.fit_transform(feature_frame, y.head(10_000))
encoded_sample[["Region_Code_response_rate_te", "Policy_Sales_Channel_response_rate_te"]].describe()

,Region_Code_response_rate_te,Policy_Sales_Channel_response_rate_te
count,10000.000000,10000.000000
mean,0.127416,0.122765
std,0.049484,0.088910
min,0.034639,0.014329
25%,0.094861,0.026184
50%,0.108186,0.156722
75%,0.192620,0.204826
max,0.214163,0.358027
